In [1]:
!pip install -q -U ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 2.7 MB/s eta 0:00:00


In [2]:
import cv2
import os
from collections import defaultdict, deque
import numpy as np
from ultralytics import YOLO
from collections import defaultdict, deque
import shutil

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
modelPath = "/content/drive/MyDrive/TrafficObj_YOLO/best.pt"

# Check that the model exists
print("Model exists:", os.path.exists(modelPath))

# Load model
model = YOLO(modelPath)

print("Model loaded successfully!")
print("Classes:", model.names)

Model exists: True
Model loaded successfully!
Classes: {0: 'Bus', 1: 'Truck', 2: 'Car', 3: 'Bicycle', 4: 'bike', 5: 'auto', 6: 'pedestrian'}


In [8]:
videoPath = "/content/drive/MyDrive/ObjectTrackerVideo/video.mp4"
print("Video exists:", os.path.exists(videoPath))

Video exists: True


In [15]:
from ultralytics import YOLO
import cv2
import os
from collections import defaultdict, deque

outputDir = "/content/traffic_tracking_counting"
os.makedirs(outputDir, exist_ok=True)

outputVideoPath = os.path.join(
    outputDir,
    "tracked_counted_output.mp4"
)


# OPEN VIDEO
cap = cv2.VideoCapture(videoPath)

if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {videoPath}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
totalFrames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("\nVideo information:")
print("Width:", width)
print("Height:", height)
print("FPS:", fps)
print("Total frames:", totalFrames)

# COUNTING LINE
# 0.60 means 60% down from the top of the video.

LINE_Y = int(height * 0.35)
print("Counting line Y:", LINE_Y)
# VIDEO WRITER
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    outputVideoPath,
    fourcc,
    fps,
    (width, height)
)
# TRACK HISTORY
# Stores previous center points for every tracking ID.
trackHistory = defaultdict(lambda: deque(maxlen=30))
previousPositions = {}
countedIDs = set()

# CLASS COUNTERS
classCounts = {
    0: 0,  # Bus
    1: 0,  # Truck
    2: 0,  # Car
    3: 0,  # Bicycle
    4: 0,  # Bike
    5: 0,  # Auto
    6: 0   # Pedestrian
}
# CLASS NAMES
classNames = {
    0: "Bus",
    1: "Truck",
    2: "Car",
    3: "Bicycle",
    4: "Bike",
    5: "Auto",
    6: "Pedestrian"
}
# PROCESS VIDEO FRAME BY FRAME
frameNumber = 0
while True:
    success, frame = cap.read()
    if not success:
        break
    frameNumber += 1
    # YOLO + BYTE TRACK
    results = model.track(
        frame,
        persist=True,
        tracker="bytetrack.yaml",
        conf=0.25,
        imgsz=640,
        device=0,
        verbose=False
    )
    result = results[0]
    # DRAW COUNTING LINE
    cv2.line(
        frame,
        (0, LINE_Y),
        (width, LINE_Y),
        (0, 255, 255),
        3
    )

    cv2.putText(
        frame,
        "COUNTING LINE",
        (20, LINE_Y - 10),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 255),
        2
    )
    # CHECK WHETHER TRACK IDs EXIST
    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        classes = result.boxes.cls.cpu().numpy().astype(int)
        trackIDs = result.boxes.id.cpu().numpy().astype(int)
        # PROCESS EACH TRACKED OBJECT
        for box, classID, trackID in zip(
            boxes,
            classes,
            trackIDs
        ):
            x1, y1, x2, y2 = map(int, box)
            centerX = int((x1 + x2) / 2)
            centerY = int((y1 + y2) / 2)
            # SAVE TRACK HISTORY
            trackHistory[trackID].append(
                (centerX, centerY)
            )
            # DRAW TRACKING TAIL
            points = list(trackHistory[trackID])
            for i in range(1, len(points)):
                cv2.line(
                    frame,
                    points[i - 1],
                    points[i],
                    (255, 0, 255),
                    3
                )
            # DRAW CENTER POINT
            cv2.circle(
                frame,
                (centerX, centerY),
                5,
                (255, 0, 255),
                -1
            )
            # DRAW BOUNDING BOX
            cv2.rectangle(
                frame,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                2
            )

            className = classNames.get(
                classID,
                str(classID)
            )
            # DRAW CLASS + TRACK ID
            label = f"{className} ID:{trackID}"
            cv2.putText(
                frame,
                label,
                (x1, max(y1 - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 255, 0),
                2
            )
            # LINE CROSSING
            if trackID in previousPositions:
                previousY = previousPositions[trackID]
                crossedLine = (
                    previousY < LINE_Y and
                    centerY >= LINE_Y
                )
                if crossedLine and trackID not in countedIDs:
                    countedIDs.add(trackID)
                    if classID in classCounts:
                        classCounts[classID] += 1
            previousPositions[trackID] = centerY

    # DRAW COUNTER PANEL
    panelX = 20
    panelY = 30
    panelWidth = 300
    panelHeight = 260

    overlay = frame.copy()
    cv2.rectangle(
        overlay,
        (panelX, panelY),
        (panelX + panelWidth, panelY + panelHeight),
        (0, 0, 0),
        -1
    )

    frame = cv2.addWeighted(
        overlay,
        0.55,
        frame,
        0.45,
        0
    )

    cv2.putText(
        frame,
        "TRAFFIC COUNT",
        (panelX + 15, panelY + 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 255),
        2
    )

    yText = panelY + 60
    for classID, className in classNames.items():
        text = f"{className}: {classCounts[classID]}"
        cv2.putText(
            frame,
            text,
            (panelX + 15, yText),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2
        )

        yText += 28
    # FRAME NUMBER
    cv2.putText(
        frame,
        f"Frame: {frameNumber}/{totalFrames}",
        (width - 280, 35),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.65,
        (255, 255, 255),
        2
    )
    # WRITE FRAME
    writer.write(frame)
    # PROGRESS
    if frameNumber % 100 == 0:
        progress = (
            frameNumber / totalFrames
        ) * 100
        print(
            f"Processed: {frameNumber}/{totalFrames} "
            f"({progress:.1f}%)"
        )
# RELEASE
cap.release()
writer.release()
print("PROCESSING COMPLETE")
print("\nFinal counts:")

for classID, className in classNames.items():
    print(
        f"{className}: "
        f"{classCounts[classID]}"
    )

print("\nOutput video:")
print(outputVideoPath)


Video information:
Width: 1280
Height: 720
FPS: 30.0
Total frames: 5108
Counting line Y: 251
Processed: 100/5108 (2.0%)
Processed: 200/5108 (3.9%)
Processed: 300/5108 (5.9%)
Processed: 400/5108 (7.8%)
Processed: 500/5108 (9.8%)
Processed: 600/5108 (11.7%)
Processed: 700/5108 (13.7%)
Processed: 800/5108 (15.7%)
Processed: 900/5108 (17.6%)
Processed: 1000/5108 (19.6%)
Processed: 1100/5108 (21.5%)
Processed: 1200/5108 (23.5%)
Processed: 1300/5108 (25.5%)
Processed: 1400/5108 (27.4%)
Processed: 1500/5108 (29.4%)
Processed: 1600/5108 (31.3%)
Processed: 1700/5108 (33.3%)
Processed: 1800/5108 (35.2%)
Processed: 1900/5108 (37.2%)
Processed: 2000/5108 (39.2%)
Processed: 2100/5108 (41.1%)
Processed: 2200/5108 (43.1%)
Processed: 2300/5108 (45.0%)
Processed: 2400/5108 (47.0%)
Processed: 2500/5108 (48.9%)
Processed: 2600/5108 (50.9%)
Processed: 2700/5108 (52.9%)
Processed: 2800/5108 (54.8%)
Processed: 2900/5108 (56.8%)
Processed: 3000/5108 (58.7%)
Processed: 3100/5108 (60.7%)
Processed: 3200/5108 

In [16]:
driveOutput = "/content/drive/MyDrive/ObjectTrackerVideo/tracked_counted_output.mp4"

shutil.copy2(
    outputVideoPath,
    driveOutput
)

print("Video saved to Google Drive:")
print(driveOutput)

Video saved to Google Drive:
/content/drive/MyDrive/ObjectTrackerVideo/tracked_counted_output.mp4
